# Kiểm tra môi trường đánh giá
Chọn kernel **Acne Advisor AI (evaluation)**. Notebook này chỉ kiểm tra cài đặt và đọc artifact; không gọi API hoặc chạy lại đánh giá nghiên cứu.

In [ ]:
import importlib.metadata as metadata
import os
import sys
from pathlib import Path

os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'

ROOT = Path.cwd().resolve()
if ROOT.name == 'evaluation':
    ROOT = ROOT.parent
assert (ROOT / 'evaluation' / 'benchmark_100.json').is_file(), 'Mở notebook từ thư mục dự án.'
sys.path.insert(0, str(ROOT)) if str(ROOT) not in sys.path else None
assert Path(sys.prefix).resolve() == (ROOT / 'evaluation' / '.venv').resolve(), 'Chọn kernel Acne Advisor AI (evaluation).'
assert sys.version_info[:2] == (3, 11)
for package in ['jupyterlab', 'ipykernel', 'openai', 'ragchecker', 'refchecker', 'spacy', 'transformers', 'torch']:
    print(f'{package}: {metadata.version(package)}')
assert metadata.version('ragchecker') == '0.1.9'
assert metadata.version('openai') == '2.38.0'
assert metadata.version('transformers').split('.')[0] == '4'
print('Python:', sys.executable)

In [ ]:
import spacy
from openai import OpenAI
from ragchecker import RAGChecker, RAGResult, RAGResults
from ragchecker.metrics import claim_recall, context_precision, f1, faithfulness

nlp = spacy.load('en_core_web_sm')
doc = nlp('This is an offline environment check.')
assert len(doc) > 0
print('RAGChecker/OpenAI imports: OK')
print('spaCy en_core_web_sm:', nlp.meta['version'])

In [ ]:
import json
import subprocess
from evaluation.formal_evaluation_support import load_evaluation_artifacts

benchmark, manifest, calibration = load_evaluation_artifacts()
assert len(benchmark['cases']) == 100
print('Benchmark:', len(benchmark['cases']), 'cases')
for path in sorted((ROOT / 'evaluation' / 'results').glob('*/raw_results.json')):
    json.loads(path.read_text(encoding='utf-8'))
    print('Readable saved output:', path.relative_to(ROOT))
result = subprocess.run([sys.executable, '-m', 'pip', 'check'], capture_output=True, text=True, check=True)
print(result.stdout.strip())
print('EVALUATION ENVIRONMENT: PASS')